In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialize the Spark Session
# "local[*]" tells Spark to use all available CPU cores on your machine
spark = SparkSession.builder \
    .appName("GeoPulse_Local_Test") \
    .master("local[*]") \
    .getOrCreate()

# 2. Create some raw mock data (DeviceID, Lat, Long, Timestamp)
mock_data = [
    ("Device_123", 37.7749, -122.4194, "08:00"),
    ("Device_456", 37.7749, -122.4194, "08:05"),
    ("Device_123", 37.7750, -122.4190, "08:10"),
    ("Device_789", 34.0522, -118.2437, "08:15"),
    ("Device_123", 37.7755, -122.4180, "08:20")
]
columns = ["DeviceID", "Lat", "Long", "Time"]

# 3. Load the data into a distributed DataFrame
df = spark.createDataFrame(mock_data, columns)

print("\n--- Raw GPS Pings ---")
df.show()

# 4. Perform a transformation: Count how many times each device pinged
ping_counts = df.groupBy("DeviceID") \
    .agg(F.count("*").alias("Total_Pings")) \
    .orderBy(F.col("Total_Pings").desc())

print("\n--- Aggregated Ping Counts ---")
# 5. Action: spark doesn't actually compute anything until you call an action like show()
ping_counts.show()

# 6. Shut down the cluster
spark.stop()


--- Raw GPS Pings ---
+----------+-------+---------+-----+
|  DeviceID|    Lat|     Long| Time|
+----------+-------+---------+-----+
|Device_123|37.7749|-122.4194|08:00|
|Device_456|37.7749|-122.4194|08:05|
|Device_123| 37.775| -122.419|08:10|
|Device_789|34.0522|-118.2437|08:15|
|Device_123|37.7755| -122.418|08:20|
+----------+-------+---------+-----+


--- Aggregated Ping Counts ---
+----------+-----------+
|  DeviceID|Total_Pings|
+----------+-----------+
|Device_123|          3|
|Device_456|          1|
|Device_789|          1|
+----------+-----------+

